In [100]:
# transform symbolic midi to wave form
import pretty_midi
# visuals in jyupter
import IPython.display as ipd
import torch 
import torch.nn as nn
import sys
sys.path.append("..")
# visualize piana roles
import libfmp.c1 
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px

In [101]:
music_path  = Path("~/python/deeplearning/music_lab/Jazz_Midi").expanduser()
print(music_path)

/Users/chenyi/python/deeplearning/music_lab/Jazz_Midi


In [102]:
midi_files = list(music_path.glob("*.mid"))

In [103]:
test_midi_song = pretty_midi.PrettyMIDI(midi_files[50])


In [104]:
musi = []
for i,intrument in enumerate(test_midi_song.instruments):
    for note in intrument.notes:
        musi.append([
            note.start,
            note.end - note.start,
            note.pitch,
            note.velocity,
            intrument.program
        ])
    musi = sorted(musi, key= lambda x : (x[0],x[2]))
musi_pd = pd.DataFrame(data = musi, columns= ("start", "duration", "pitch", "velocity", "instrument"))
        


In [105]:
musi_pd.head(30)
musi_pd.tail(30)

,start,duration,pitch,velocity,instrument
406,106.153740,0.461538,62,95,25
407,106.615278,0.461538,50,95,25
408,106.615278,0.461538,59,95,25
409,107.076816,0.307692,40,95,25
410,107.076816,0.307692,61,95,25
411,107.384508,0.153846,57,95,25
412,107.538354,0.307692,50,95,25
413,107.846046,0.153846,59,95,25
414,107.999892,0.461538,47,95,25
415,108.461430,0.461538,50,95,25


In [106]:
musi_pd["duration"].sort_values(ascending= False)

435    2.307690
434    2.307690
433    2.307690
432    2.307690
304    0.461538
         ...   
72     0.153846
105    0.153846
96     0.153846
196    0.153846
285    0.153846
Name: duration, Length: 436, dtype: float64

In [315]:
def event_to_token(event):
    position, duration, pitch, velocity, program, role = event

    duration_id = (
        max(1, min(int(duration), 288)) - 1
    )

    return [
        int(position),
        duration_id,
        int(pitch),
        int(velocity),
        int(program),
        int(role),
    ]

In [316]:
def get_bar_informations(bar, STEPS_PER_BAR = 48):
    MAX_NOTES_PER_ROLE = 48
    # density order[ drums, bass, piano, guitar, ]
    chroma = torch.zeros(12)
    density = torch.zeros(6)
    energy = torch.zeros(1)
    rhythm = torch.zeros(48)
    longest_overhang = torch.zeros(6)

    for event in bar:
        position, duration, pitch, velocity, program, role = event_to_token(event)

        
        density[role] += duration

        energy[0] += velocity*duration

        if role!= 0:
            pitch_class = pitch % 12
            chroma[pitch_class] += duration * velocity

        add = 1
        for i in range(position, position+duration):
            if i >=48:
                longest_overhang[role] = 1
                break
            multiplier = 0.75
            rhythm[i] += add
            add*= multiplier


    # normalize the chroma vector
    if chroma.max() > 0:
        chroma /= chroma.max()
    # normalize energy
    energy[0] /= (STEPS_PER_BAR*127) 
    # normalize density
    if density.max() >0:
        density /= (MAX_NOTES_PER_ROLE) 

    rhythm /= 6

    # Cap values above 1 while retaining values below 1.
    energy.clamp_(min=0.0, max=1.0)
    density.clamp_(min=0.0, max=1.0)
    rhythm.clamp_(min=0.0, max=1.0)
    return torch.cat([chroma,density,energy,rhythm,longest_overhang])

In [317]:
def get_instrument_role(instrument):
    if instrument.is_drum:
        return 0  # Drums

    program = instrument.program

    if 32 <= program <= 39:
        return 1  # Bass

    if 0 <= program <= 23:
        return 2  # Piano, keyboard, organ

    if 24 <= program <= 31:
        return 3  # Guitar/melody

    if 56 <= program <= 79:
        return 4  # Brass, saxophone, flute

    return 5      # Strings, pads, and other instruments

In [318]:

def song_to_bar(midi_song):

    STEPS_PER_BEAT = 12
    STEPS_PER_BAR = 48   

    song = {}
    for i,instrument in enumerate(midi_song.instruments):
        role = get_instrument_role(instrument)
            # every bar is divided into 48 position
        for note in instrument.notes:
                start_tick = midi_song.time_to_tick(note.start)
                end_tick = midi_song.time_to_tick(note.end)

                start_step = round(start_tick/midi_song.resolution*STEPS_PER_BEAT)
                start_position = start_step%STEPS_PER_BAR
                bar_number = start_step//STEPS_PER_BAR
                end_step = round(end_tick/midi_song.resolution*STEPS_PER_BEAT)

                # append events used for next note prediction
                if bar_number in song:
                    song[bar_number]["events"].append([
                        start_position,
                        max(1,min(end_step-start_step, 288)),
                        note.pitch,
                        note.velocity,
                        instrument.program,
                        role
                    ])
                else:
                    song[bar_number] = {"events":[[
                        start_position,
                        max(1,min(end_step-start_step, 288)),
                        note.pitch,
                        note.velocity,
                        instrument.program,
                        role                     
                    ]]}

                
    for key in song.keys():
        # sort by start/ role/ pitch
        song[key]["events"] = sorted(song[key]["events"], key= lambda x : (x[0],x[5],x[2]))

        # append in information about bar for the composer
        song[key]["bar_info"] = get_bar_informations(song[key]["events"])
       
    return song

             
            


In [319]:
song = pretty_midi.PrettyMIDI(midi_files[16])
song = song_to_bar(song)
print(song)

{0: {'events': [[12, 12, 43, 95, np.int64(28), 3], [24, 12, 45, 95, np.int64(28), 3], [36, 12, 47, 95, np.int64(28), 3]], 'bar_info': tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
        1.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.6875, 0.0000, 0.0000,
        0.5143, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.1667, 0.1250, 0.0938, 0.0703, 0.0527,
        0.0396, 0.0297, 0.0222, 0.0167, 0.0125, 0.0094, 0.0000, 0.1667, 0.1250,
        0.0938, 0.0703, 0.0527, 0.0396, 0.0297, 0.0222, 0.0167, 0.0125, 0.0094,
        0.0000, 0.1667, 0.1250, 0.0938, 0.0703, 0.0527, 0.0396, 0.0297, 0.0222,
        0.0167, 0.0125, 0.0094, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000])}, 1: {'events': [[0, 12, 48, 95, np.int64(28), 3], [12, 6, 52, 95, np.int64(28), 3], [12, 6, 55, 95, np.int64(28), 3], [12, 6, 64, 95, np.int64(28), 3], [18, 6, 60, 95, np.int64(28), 3], [24, 6, 43, 95, np.int

In [ ]:
# embedding plan
# position:  0–47
# duration:  0–287 -> 6 bars
# pitch:     0–127
# velocity:  0–7
# program:   0–127
# role:      0–5
# drum:      0–1

# "chroma": ...,        # 12 values
# "density": ...,       # 6 values
# "energy": ...,        # 1 value
# "rhythm": ...         # 48 values
# overhang -> 6

In [320]:
def load_dataset():
    music_file = []
    # load all the dataset sorted by start time into list sorted by start time
    for i in range(len(midi_files)):
        try:
            midi_song = pretty_midi.PrettyMIDI(midi_files[i])
            song = song_to_bar(midi_song)
            music_file.append(song)
        except:
            print(f"bad music as position{i}")
    return music_file


In [321]:
music_file = load_dataset()

/Users/chenyi/python/deeplearning/.venv/lib/python3.11/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


bad music as position34
bad music as position36
bad music as position62
bad music as position70
bad music as position122
bad music as position143
bad music as position169
bad music as position192
bad music as position200
bad music as position271
bad music as position370
bad music as position427
bad music as position503
bad music as position510
bad music as position513
bad music as position538
bad music as position546
bad music as position745
bad music as position767


In [137]:
class LSTM_composer(nn.Module):
    def __init__(self, input_dim = 73, hidden_dim = 256, output_dim = 73):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.forgetgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.forgetgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.inputgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.inputgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.candidate_ltm_h = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.candidate_ltm_x = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.outputgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.outputgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.output_w_b = nn.Linear(in_features= hidden_dim, out_features= output_dim)

        self.stm = None
        self.ltm = None


    def forward(self, input):

        # embedding input
        batch_size = input.shape[0]

        # intialize or reintialize when we get a new input/ first time training the model
        if self.stm is None or self.stm.shape[0] != batch_size:
            self.stm = torch.zeros(
                batch_size,
                self.hidden_dim,

            )
            self.ltm = torch.zeros(
                batch_size,
                self.hidden_dim,

            )

        # fowarding math
        forgetgate = nn.functional.sigmoid(self.forgetgateh(self.stm) + self.forgetgatex(input))
        inputgate = nn.functional.sigmoid(self.inputgateh(self.stm) + self.inputgatex(input))
        candidate_ltm = nn.functional.tanh(self.candidate_ltm_h(self.stm) + self.candidate_ltm_x(input))
        outputgate = nn.functional.sigmoid(self.outputgateh(self.stm) + self.outputgatex(input))

        self.ltm = self.ltm * forgetgate + inputgate * candidate_ltm 
        self.stm = outputgate * nn.functional.tanh(self.ltm)

        output = torch.sigmoid(self.output_w_b(self.stm))

        return output

    def reset(self, input):
        batch_size = input.shape[0]
        self.stm = torch.zeros(
            batch_size,
            self.hidden_dim,
            )
        self.ltm = torch.zeros(
            batch_size,
            self.hidden_dim,
        )

In [156]:
class LSTM_next_note_model(nn.Module):
    def __init__(self, input_dim = 416, hidden_dim = 512):
        super().__init__()
        self.hidden_dim = hidden_dim


        self.embedding_start = nn.Embedding(
            num_embeddings= 48,
            embedding_dim= 32
        )
        self.embedding_duration = nn.Embedding(
            num_embeddings= 288,
            embedding_dim= 32
        )
        self.embedding_pitch = nn.Embedding(
            num_embeddings= 128,
            embedding_dim= 128
        )
        self.embedding_velocity = nn.Embedding(
            num_embeddings= 128,
            embedding_dim= 16
        )
        self.embedding_program = nn.Embedding(
            num_embeddings= 128,
            embedding_dim= 128
        )
        self.embedding_role = nn.Embedding(
            num_embeddings= 6,
            embedding_dim= 16
        )
        self.bar_plan_encoder = nn.Sequential(
            nn.Linear(73, 64),
            nn.ReLU(),
        )


        self.forgetgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.forgetgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.inputgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.inputgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.candidate_ltm_h = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.candidate_ltm_x = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.outputgateh = nn.Linear(in_features= hidden_dim, out_features= hidden_dim, bias= False)
        self.outputgatex = nn.Linear(in_features= input_dim, out_features= hidden_dim)

        self.position_head = nn.Linear(hidden_dim, 48)
        self.duration_head = nn.Linear(hidden_dim, 288)
        self.pitch_head = nn.Linear(hidden_dim, 128)
        self.velocity_head = nn.Linear(hidden_dim, 128)
        self.program_head = nn.Linear(hidden_dim, 128)
        self.role_head = nn.Linear(hidden_dim, 6)

        self.stm = None
        self.ltm = None

    
    def forward(self, input, bar_plan):

        # embedding input
        batch_size = input.shape[0]

        input_start =  self.embedding_start(input[:, 0].long())
        input_duration =  self.embedding_duration(input[:, 1].long())
        input_pitch =  self.embedding_pitch(input[:,2].long())
        input_velocity =  self.embedding_velocity(input[:,3].long())
        input_program =  self.embedding_program(input[:,4].long())
        input_role =  self.embedding_role(input[:, 5].long())
        note_embedding = torch.cat((input_start, input_duration, input_pitch, input_velocity, input_program, input_role),dim = 1)
        plan_embedding = self.bar_plan_encoder(bar_plan)
        input = torch.cat((note_embedding, plan_embedding), dim=1)

        # intialize or reintialize when we get a new input/ first time training the model
        if self.stm is None or self.stm.shape[0] != batch_size:
            self.stm = torch.zeros(
                batch_size,
                self.hidden_dim,

            )
            self.ltm = torch.zeros(
                batch_size,
                self.hidden_dim,

            )

        # fowarding math
        forgetgate = nn.functional.sigmoid(self.forgetgateh(self.stm) + self.forgetgatex(input))
        inputgate = nn.functional.sigmoid(self.inputgateh(self.stm) + self.inputgatex(input))
        candidate_ltm = nn.functional.tanh(self.candidate_ltm_h(self.stm) + self.candidate_ltm_x(input))
        outputgate = nn.functional.sigmoid(self.outputgateh(self.stm) + self.outputgatex(input))

        self.ltm = self.ltm * forgetgate + inputgate * candidate_ltm 
        self.stm = outputgate * nn.functional.tanh(self.ltm)
        outputs = {
            "position": self.position_head(self.stm),
            "duration": self.duration_head(self.stm),
            "pitch": self.pitch_head(self.stm),
            "velocity": self.velocity_head(self.stm),
            "program": self.program_head(self.stm),
            "role": self.role_head(self.stm),
        }

        return outputs

    def reset(self, input):
        batch_size = input.shape[0]
        self.stm = torch.zeros(
            batch_size,
            self.hidden_dim,
            )
        self.ltm = torch.zeros(
            batch_size,
            self.hidden_dim,
        )

In [243]:
music_file = load_dataset()

/Users/chenyi/python/deeplearning/.venv/lib/python3.11/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


bad music as position34
bad music as position36
bad music as position62
bad music as position70
bad music as position122
bad music as position143
bad music as position169
bad music as position192
bad music as position200
bad music as position271
bad music as position370
bad music as position427
bad music as position503
bad music as position510
bad music as position513
bad music as position538
bad music as position546
bad music as position745
bad music as position767


In [201]:
def get_training_data_bar(seqlength=64, batch_size=256):

    bar_input = []
    bar_target = []

    while len(bar_input) < batch_size:

        music_index = np.random.randint(0, len(music_file))
        song = music_file[music_index]

        available_bars = sorted(song.keys())

        # need seq_length + 1 bars
        # because targets are shifted by one
        if len(available_bars) < seqlength + 1:
            continue

        start = np.random.randint(
            0,
            len(available_bars) - seqlength
        )

        selected_bars = available_bars[
            start : start + seqlength + 1
        ]

        temp_input = []
        temp_target = []

        for j in range(seqlength):

            current_bar = selected_bars[j]
            next_bar = selected_bars[j + 1]

            temp_input.append(
                song[current_bar]["bar_info"]
            )

            temp_target.append(
                song[next_bar]["bar_info"]
            )

        bar_input.append(
            torch.stack(temp_input)
        )

        bar_target.append(
            torch.stack(temp_target)
        )

    bar_input = torch.stack(bar_input)
    bar_target = torch.stack(bar_target)

    return bar_input, bar_target

    

In [311]:
def get_note_training_data(
    seq_length=100,
    batch_size=64,
):
    note_inputs = []
    note_targets = []
    bar_plans = []

    while len(note_inputs) < batch_size:
        music_index = np.random.randint(
            0,
            len(music_file),
        )

        song = music_file[music_index]

        all_notes = []
        all_plans = []

        for bar_number in sorted(song.keys()):
            events = song[bar_number]["events"]

            if len(events) == 0:
                continue

            event_tokens = torch.tensor(
                [
                    event_to_token(event)
                    for event in events
                ],
                dtype=torch.long,
            )

            bar_plan = song[bar_number]["bar_info"]

            all_notes.append(event_tokens)

            for _ in range(len(event_tokens)):
                all_plans.append(bar_plan)

        if not all_notes:
            continue

        all_notes = torch.cat(
            all_notes,
            dim=0,
        )

        all_plans = torch.stack(
            all_plans,
            dim=0,
        )

        if len(all_notes) < seq_length + 1:
            continue

        index = np.random.randint(
            0,
            len(all_notes) - seq_length,
        )

        note_inputs.append(
            all_notes[index:index + seq_length]
        )

        note_targets.append(
            all_notes[
                index + 1:index + seq_length + 1
            ]
        )

        # Target plans, not input plans.
        bar_plans.append(
            all_plans[
                index + 1:index + seq_length + 1
            ]
        )

    return (
        torch.stack(note_inputs),
        torch.stack(note_targets),
        torch.stack(bar_plans),
    )



In [322]:
note_inputs, note_targets, plans = (
    get_note_training_data(
        seq_length=100,
        batch_size=4,
    )
)

print(note_inputs.shape)   # [4, 100, 6]
print(note_targets.shape)  # [4, 100, 6]
print(plans.shape)         # [4, 100, 73]

print(note_inputs[:, :, 1].min())  # >= 0
print(note_inputs[:, :, 1].max())  # <= 287

torch.Size([4, 100, 6])
torch.Size([4, 100, 6])
torch.Size([4, 100, 73])
tensor(0)
tensor(142)


In [159]:
def calculate_composer_loss(prediction, target):
    # Layout:
    # 0:12   chroma
    # 12:18  density
    # 18:19  energy
    # 19:67  rhythm
    # 67:73  overhang

    predicted_chroma = prediction[:, 0:12]
    predicted_density = prediction[:, 12:18]
    predicted_energy = prediction[:, 18:19]
    predicted_rhythm = prediction[:, 19:67]
    predicted_overhang = prediction[:, 67:73]

    target_chroma = target[:, 0:12]
    target_density = target[:, 12:18]
    target_energy = target[:, 18:19]
    target_rhythm = target[:, 19:67]
    target_overhang = target[:, 67:73]

    loss_chroma = nn.functional.mse_loss(
        predicted_chroma,
        target_chroma,
    )

    loss_density = nn.functional.mse_loss(
        predicted_density,
        target_density,
    )

    loss_energy = nn.functional.mse_loss(
        predicted_energy,
        target_energy,
    )

    loss_rhythm = nn.functional.mse_loss(
        predicted_rhythm,
        target_rhythm,
    )

    # Overhang consists of binary 0/1 values.
    loss_overhang = nn.functional.binary_cross_entropy(
        predicted_overhang,
        target_overhang,
    )

    loss = (
        2.0 * loss_chroma
        + 1.0 * loss_density
        + 0.5 * loss_energy
        + 1.5 * loss_rhythm
        + 0.5 * loss_overhang
    )

    return loss

In [160]:
def calculate_note_loss(prediction, target):
    # Target layout:
    # 0   position:  0–47
    # 1   duration:  0–287
    # 2   pitch:     0–127
    # 3   velocity:  0–127
    # 4   program:   0–127
    # 5   role:      0–5

    target_position = target[:, 0].long()
    target_duration = target[:, 1].long()
    target_pitch = target[:, 2].long()
    target_velocity = target[:, 3].long()
    target_program = target[:, 4].long()
    target_role = target[:, 5].long()

    loss_position = nn.functional.cross_entropy(
        prediction["position"],
        target_position,
    )

    loss_duration = nn.functional.cross_entropy(
        prediction["duration"],
        target_duration,
    )

    loss_pitch = nn.functional.cross_entropy(
        prediction["pitch"],
        target_pitch,
    )

    loss_velocity = nn.functional.cross_entropy(
        prediction["velocity"],
        target_velocity,
    )

    loss_program = nn.functional.cross_entropy(
        prediction["program"],
        target_program,
    )

    loss_role = nn.functional.cross_entropy(
        prediction["role"],
        target_role,
    )

    loss = (
        1.5 * loss_position
        + 1.0 * loss_duration
        + 2.0 * loss_pitch
        + 0.5 * loss_velocity
        + 1.0 * loss_program
        + 0.5 * loss_role
    )

    return loss

In [294]:
composer_model = LSTM_composer()
optimizer = torch.optim.AdamW(
    composer_model.parameters(),
    lr= 0.001,
    weight_decay= 0.0001
)


def bar_training_loop(batch_size=256, iteration = 512):
    composer_loss_graph = []
    batch_size = batch_size
    seqlength = 64
    j = 0
    bar_input, bar_target = get_training_data_bar(
        seqlength,
        batch_size
    )
    for epoch in tqdm(range(iteration)):

        composer_model.train()
        composer_model.reset(bar_input)

        # get new input & target in each 5 iteration
        j+=1
        if (j+1) % 5 ==0:
            bar_input, bar_target = get_training_data_bar(
            seqlength,
            batch_size
        )

        # loss calculation
        total_loss = 0
        for i in range(bar_input.shape[1]):
            logit = composer_model(bar_input[:, i])
            loss = calculate_composer_loss(logit, bar_target[:,i])
            total_loss += loss

            # stepping every 30 timeslot
            if (i + 1) % 30 == 0 or i == seqlength -1:
                total_loss = total_loss / 30
                optimizer.zero_grad()
                total_loss.backward()
                
                torch.nn.utils.clip_grad_norm_(
                    composer_model.parameters(),
                    max_norm=1.0
                )
                optimizer.step()
                composer_model.stm = composer_model.stm.detach()
                composer_model.ltm = composer_model.ltm.detach()
                composer_loss_graph.append(total_loss)
                total_loss = 0
  

    return composer_loss_graph

In [325]:
note_model = LSTM_next_note_model()
note_optimizer = torch.optim.AdamW(
    note_model.parameters(),
    lr= 0.001,
    weight_decay= 0.0001
)

def note_training_loop( batch_size=128, iteration=256):

    sequence_length=200
    batch_size = batch_size
    note_loss_graph = []
    note_input, note_targets, bar_plan = get_note_training_data(
            sequence_length,
            batch_size
        )
    print(type(note_input))
    for epoch in tqdm(range(iteration)):
        note_model.train()

        # Select new bars every five epochs.
        if epoch > 0 and epoch % 1 == 0:
            note_input, note_targets, bar_plan = get_note_training_data(
                    sequence_length,
                    batch_size
                )

        note_model.reset(note_input[:, 0])

        total_loss = 0.0

        for i in range(note_input.shape[1]):
            current_note = note_input[:, i]
            target_note = note_targets[:, i]
            current_plan = bar_plan[:,i]

            outputs = note_model.forward(
                    current_note,
                    current_plan
            )

            loss = calculate_note_loss(
                outputs,
                target_note
            )

            total_loss += loss
            note_loss_graph.append(
                    loss
                )
            if(i+1)%50 ==0 or i == sequence_length-1:
        
                note_optimizer.zero_grad()
                total_loss/= 50
                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    note_model.parameters(),
                    max_norm=1.0,
                )
                note_optimizer.step()
                

                total_loss = 0
                note_model.stm = note_model.stm.detach()
                note_model.ltm = note_model.ltm.detach()


    return note_loss_graph

In [324]:
bar_loss_graph = bar_training_loop()


100%|██████████| 512/512 [00:56<00:00,  9.08it/s]


In [326]:
note_loss_graph = note_training_loop()

<class 'torch.Tensor'>


100%|██████████| 256/256 [04:25<00:00,  1.04s/it]


In [327]:
loss_values = [
    loss.item() if torch.is_tensor(loss) else float(loss)
    for loss in bar_loss_graph
]

loss_data = pd.DataFrame({
    "training_step": np.arange(len(loss_values)),
    "loss": loss_values,
})

fig = px.scatter(
    loss_data,
    x="training_step",
    y="loss",
    title="Composer training loss",
)

fig.show()

In [328]:
loss_values = [
    loss.item() if torch.is_tensor(loss) else float(loss)
    for loss in note_loss_graph
]

loss_data = pd.DataFrame({
    "training_step": np.arange(len(loss_values)),
    "loss": loss_values,
})

fig = px.scatter(
    loss_data,
    x="training_step",
    y="loss",
    title="note training loss",
)

fig.show()

In [329]:
torch.save(
    composer_model.state_dict(),
    "midi_composer_model.pth"
)

In [330]:
torch.save(
    note_model.state_dict(),
    "midi_note_model.pth"
)

In [331]:
def sample_top_k(logits, temperature=0.8, top_k=8):
    """
    logits shape: [batch_size, number_of_categories]
    returns:      [batch_size, 1]
    """

    temperature = max(float(temperature), 1e-5)

    logits = logits / temperature

    top_k = min(top_k, logits.shape[1])

    top_values, top_indices = torch.topk(
        logits,
        k=top_k,
        dim=1,
    )

    probabilities = torch.softmax(
        top_values,
        dim=1,
    )

    sampled_top_index = torch.multinomial(
        probabilities,
        num_samples=1,
    )

    sampled_value = torch.gather(
        top_indices,
        dim=1,
        index=sampled_top_index,
    )

    return sampled_value

In [334]:
    PROGRAM_BY_ROLE = {
    0: 0,   # drums
    1: 32,  # bass
    2: 0,   # piano
    3: 24,  # guitar
    4: 65,  # saxophone
    5: 48,  # strings/other
    }

In [392]:
def get_prediction(
    seed_notes,
    bar_history,
    length=1000,
    temperature=0.8,
    top_k=8,
):
    """
    seed_notes:
        [1, seed_length, 6]

    bar_history:
        [1, number_of_previous_bars, 73]

    Returned event layout:
        [
            position,
            duration_steps,
            pitch,
            velocity,
            program,
            role,
        ]
    """

    generated_notes = []

    note_model.eval()
    composer_model.eval()

    device = next(note_model.parameters()).device

    seed_notes = seed_notes.to(device)
    bar_history = bar_history.to(device)

    # Reset both model memories.
    note_model.reset(seed_notes[:, 0])
    composer_model.reset(bar_history[:, 0])

    with torch.no_grad():
        # Run previous bar plans through the composer.
        predicted_next_plan = None

        for bar_index in range(bar_history.shape[1]):
            predicted_next_plan = composer_model(
                bar_history[:, bar_index]
            )

        # The seed notes currently belong to the last real bar.
        current_bar_plan = bar_history[:, -1]

        # Warm up the note LSTM using the seed.
        output = None

        for note_index in range(seed_notes.shape[1]):
            output = note_model(
                seed_notes[:, note_index],
                current_bar_plan,
            )

        current_note = seed_notes[:, -1]
        current_position = int(
            current_note[0, 0].item()
        )

        for _ in range(length):
            next_position = sample_top_k(
                output["position"],
                temperature=0.5,
                top_k=8,
            )

            next_duration_id = sample_top_k(
                output["duration"],
                temperature=0.5,
                top_k=8,
            )

            next_pitch = sample_top_k(
                output["pitch"],
                temperature=0.7,
                top_k=8,
            )
            next_role = sample_top_k(
                output["role"],
                temperature=0.4,
                top_k=4,
            )
            next_velocity = sample_top_k(
                output["velocity"],
                temperature=temperature,
                top_k=8,
            )

            role_value = int(next_role.item())

            next_program = torch.tensor(
                [[PROGRAM_BY_ROLE[role_value]]],
                device=next_role.device,
                dtype=torch.long,
            )

            next_role = sample_top_k(
                output["role"],
                temperature=0.4,
                top_k=min(top_k, 6),
            )

            next_note = torch.cat([
                next_position,
                next_duration_id,
                next_pitch,
                next_velocity,
                next_program,
                next_role,
            ], dim=1).long()

            position_value = int(
                next_position[0, 0].item()
            )

            # A position reset signals a new bar.
            if position_value < current_position:
                current_bar_plan = predicted_next_plan

                # Predict the plan following this new bar.
                predicted_next_plan = composer_model(
                    current_bar_plan
                )

            duration_steps = (
                int(next_duration_id[0, 0].item()) + 1
            )

            generated_notes.append([
                position_value,
                duration_steps,
                int(next_pitch[0, 0].item()),
                int(next_velocity[0, 0].item()),
                int(next_program[0, 0].item()),
                int(next_role[0, 0].item()),
            ])

            current_note = next_note
            current_position = position_value

            output = note_model(
                current_note,
                current_bar_plan,
            )

    return generated_notes

In [393]:

def generated_notes_to_midi(
    generated_notes,
    filename="generated_jazz.mid",
    tempo=120,
    steps_per_beat=12,
    beats_per_bar=4,
):
    midi = pretty_midi.PrettyMIDI(
        initial_tempo=tempo
    )

    instruments = {}

    current_bar = 0
    previous_position = None
    notes_saved = 0

    seconds_per_beat = 60.0 / tempo

    for event in generated_notes:
        (
            position,
            duration_steps,
            pitch,
            velocity,
            program,
            role,
        ) = event

        position = int(position)
        duration_steps = max(1, int(duration_steps))
        pitch = int(pitch)
        velocity = int(velocity)
        program = int(program)
        role = int(role)

        # Skip invalid model outputs safely.
        if not 0 <= position < 48:
            continue

        if not 0 <= pitch <= 127:
            continue

        if not 0 <= velocity <= 127:
            continue

        if not 0 <= program <= 127:
            continue

        # A lower position indicates that a new bar began.
        if (
            previous_position is not None
            and position < previous_position
        ):
            current_bar += 1

        is_drum = role == 0

        instrument_key = role
        program = PROGRAM_BY_ROLE[role]

        instrument = pretty_midi.Instrument(
            program=program,
            is_drum=(role == 0),
            name=f"Role {role}",
        )

        if instrument_key not in instruments:
            instrument = pretty_midi.Instrument(
                program=program,
                is_drum=is_drum,
                name=f"Role {role} Program {program}",
            )

            instruments[instrument_key] = instrument

        instrument = instruments[instrument_key]

        start_beat = (
            current_bar * beats_per_bar
            + position / steps_per_beat
        )

        duration_beats = (
            duration_steps / steps_per_beat
        )

        start_seconds = (
            start_beat * seconds_per_beat
        )

        end_seconds = (
            start_seconds
            + duration_beats * seconds_per_beat
        )

        note = pretty_midi.Note(
            velocity=max(1, min(velocity, 127)),
            pitch=max(0, min(pitch, 127)),
            start=float(start_seconds),
            end=float(end_seconds),
        )

        instrument.notes.append(note)

        previous_position = position
        notes_saved += 1

    for instrument in instruments.values():
        if instrument.notes:
            instrument.notes.sort(
                key=lambda note: (
                    note.start,
                    note.pitch,
                )
            )

            midi.instruments.append(instrument)

    midi.write(filename)

    print(
        f"Saved {notes_saved} notes to {filename}"
    )
    print(
        f"Length: {midi.get_end_time():.2f} seconds"
    )

    return midi

In [407]:
seed_notes, _, seed_plans = (
    get_note_training_data(
        seq_length=100,
        batch_size=1,
    )
)

current_bar_history = seed_plans[:, -1:, :]

generated = get_prediction(
    seed_notes=seed_notes,
    bar_history=current_bar_history,
    length=1000,
)

print(generated[:10])
generated_notes_to_midi(generated)

[[20, 12, 36, 90, 32, 1], [20, 12, 55, 70, 0, 4], [24, 12, 60, 70, 0, 0], [24, 12, 51, 90, 0, 0], [24, 12, 31, 90, 32, 1], [24, 36, 43, 90, 65, 4], [24, 36, 44, 90, 65, 0], [36, 36, 51, 80, 0, 0], [36, 36, 53, 70, 0, 2], [0, 36, 60, 100, 0, 2]]
Saved 1000 notes to generated_jazz.mid
Length: 105.00 seconds
